## MIG Cement Demand Forecasting
### Data Ingestion and Data Understanding

### Midlands Infrastructure Group (MIG)

This notebook represents the first stage of the Cement Demand Forecasting project.

The purpose of this notebook is to:

1. Connect to the MIG SQLite database.
2. Identify and inspect the available database tables.
3. Load the operational cement records into Python.
4. Understand the structure and grain of the dataset.
5. Document all variables and their business meaning.
6. Examine the relationships between the database tables.
7. Identify the forecasting target and potential predictor variables.
8. Perform preliminary structural and data-quality checks before cleaning, EDA, feature engineering, or modeling.

No forecasting model is developed in this notebook. The emphasis is on understanding the data and its relationship to the business problem.

### Business Context

Midlands Infrastructure Group (MIG) is a UK civil engineering and construction company operating multiple active construction sites.

Cement is a mission-critical input across MIG's projects. However, cement requirements vary considerably because of factors such as:

- changing construction and pour schedules,
- site-specific demand patterns,
- weather conditions,
- inventory availability,
- deliveries, and
- silo storage constraints.

MIG currently relies heavily on rolling construction schedules and manual planning processes. This can contribute to supply-demand mismatches.

Two major inventory problems can result:

**Stockouts**

A site may not have enough cement available when a scheduled pour needs to take place. This can delay construction activities and leave labour and equipment idle.

**Overstocking**

A site may receive or hold more cement than it requires, increasing silo utilization, tying up working capital, and creating potential material waste.

The central business problem is therefore:

> **How can MIG predict future cement demand at each construction site early enough to make better inventory and procurement decisions?**

## Project Purpose

The purpose of the project is to develop a data-driven cement demand forecasting and inventory decision-support system.

Historical cement consumption, planned pour schedules, weather conditions and inventory information will be used to estimate future cement demand across MIG's construction sites.

The resulting forecasts will subsequently support:

- inventory planning,
- stockout prevention,
- silo utilization management,
- reorder decisions,
- procurement planning, and
- operational visibility.

The forecasting component will therefore serve as the foundation for a broader inventory management system.

## Project Objectives

The project has four major business objectives.

### Forecast Accuracy

Develop a forecasting model capable of predicting site-level cement demand up to 8 weeks ahead.

Target:

**MAPE ≤ 15%**

---

### Pour Readiness

Reduce the probability that scheduled construction pours cannot proceed because cement is unavailable.

Target:

**≥ 98% pour readiness**

---

### Inventory Efficiency

Use forecasts to improve inventory utilization and reduce unnecessary cement stock.

Targets include:

- 20% improvement in inventory utilization efficiency.
- 30% reduction in material write-offs.

---

### Decision Visibility

Provide operations managers with an interactive dashboard containing:

- demand forecasts,
- inventory positions,
- silo utilization,
- stockout risks, and
- reorder alerts.

## Overall Data Science Workflow

The wider project follows seven stages:

### Stage 1 — Data Ingestion and Cleaning
Load the SQLite data, validate its structure and resolve data-quality problems.

### Stage 2 — Exploratory Data Analysis
Understand cement demand across sites, cement types, time periods, weather conditions and inventory levels.

### Stage 3 — Feature Engineering
Create useful forecasting variables such as demand lags, rolling averages, calendar variables and other operational indicators.

### Stage 4 — Forecast Model Development
Develop and compare forecasting approaches capable of predicting future cement consumption.

### Stage 5 — Inventory Simulation
Combine demand forecasts with inventory levels, deliveries and silo constraints to estimate future inventory positions.

### Stage 6 — Dashboard Development
Develop an interactive decision-support dashboard containing demand forecasts, inventory projections and reorder alerts.

### Stage 7 — Validation and Deployment
Evaluate forecast performance, deploy the solution and monitor model accuracy over time.

This notebook focuses only on **Stage 1: Data Ingestion and Understanding**.

## Operations Table — Data Dictionary

| Variable | Type | Unit | Description | Analytical Role |
|---|---|---|---|---|
| `date` | String / Date | Date | Calendar date associated with the operational observation | Time index |
| `site_id` | String | — | Unique identifier for a construction site | Site identifier |
| `cement_type` | String | — | Cement grade/category associated with the observation | Product identifier |
| `planned_pour_tonnes` | Float | tonnes | Cement quantity expected to be required according to the planned construction pour | Operational predictor |
| `consumed_tonnes` | Float | tonnes | Actual quantity of cement consumed during the day | **Primary forecasting target** |
| `opening_inventory_tonnes` | Float | tonnes | Cement inventory available at the beginning of the day | Inventory state |
| `deliveries_tonnes` | Float | tonnes | Cement delivered to the site during the day | Inventory inflow |
| `closing_inventory_tonnes` | Float | tonnes | Cement remaining at the end of the day | Inventory state |
| `rain_mm` | Float | mm | Daily rainfall associated with the site | Weather variable |
| `avg_temp_c` | Float | °C | Average daily temperature associated with the site | Weather variable |
| `silo_capacity` | Integer | tonnes | Maximum cement storage capacity at the site | Physical inventory constraint |

## Business Interpretation of the Core Variables

The variables can be grouped into six logical categories.

###  Identification Variables

`date`

Identifies when the observation occurred and provides the temporal structure required for time-series forecasting.

`site_id`

Identifies the construction site. Cement demand may vary considerably between sites because sites differ in their projects, schedules, capacities and operating conditions.

`cement_type`

Identifies the cement grade being consumed. The project therefore involves not only multiple sites but also multiple cement products.

---

### Planned Demand

`planned_pour_tonnes`

Represents the cement requirement expected from the construction schedule.

It can be interpreted as:

**What the project team expected to use.**

---

### Actual Demand

`consumed_tonnes`

Represents the quantity of cement actually consumed.

It can be interpreted as:

**What the construction site actually used.**

This is the principal outcome that the forecasting model is intended to predict.

---

### Inventory Variables

`opening_inventory_tonnes`

Cement available at the beginning of the day.

`closing_inventory_tonnes`

Cement remaining after deliveries and consumption have occurred.

---

### Supply Variable

`deliveries_tonnes`

Cement received by the site during the day.

Deliveries increase the quantity available for site operations.

---

### External Conditions

`rain_mm`

Daily rainfall associated with the construction site.

`avg_temp_c`

Average daily temperature associated with the site.

Weather conditions are included because they may help explain deviations between scheduled and actual construction activity.

---

### Physical Constraint

`silo_capacity`

Maximum quantity of cement that can be stored at the site.

Unlike demand variables, silo capacity represents a physical operational constraint and will be especially important during inventory optimization.

## Inventory Balance Relationship

The dataset contains an important operational accounting relationship.

For each operational record:

**Closing Inventory = Opening Inventory + Deliveries - Consumption**

This relationship connects four important variables:

Opening Inventory
        +
Deliveries
        -
Consumption
        =
Closing Inventory

Example:

Opening inventory = 80 tonnes  
Deliveries = 30 tonnes  
Consumption = 45 tonnes  

Expected closing inventory:

80 + 30 - 45 = 65 tonnes

This relationship provides an important data-integrity check.

During data cleaning, the recorded closing inventory will be compared with the inventory level implied by this equation.

## Forecasting Target

The primary forecasting target is:

### `consumed_tonnes`

This variable represents actual cement demand.

The principal forecasting question is therefore:

> **How much cement will a particular site consume in a future period?**

More specifically, the project aims to generate demand forecasts across sites for horizons extending up to eight weeks.

The model will eventually attempt to learn future cement demand from information such as:

- historical consumption,
- planned pours,
- site information,
- cement type,
- weather conditions,
- calendar patterns, and
- engineered historical demand features.

## Planned Demand vs Actual Demand

Two variables must be carefully distinguished:

### Planned Pour

`planned_pour_tonnes`

Represents expected cement requirements according to the construction schedule.

### Actual Consumption

`consumed_tonnes`

Represents cement actually consumed.

The relationship between these variables is important because actual construction activity may differ from scheduled activity.

For example:

| Planned Pour | Actual Consumption | Possible Interpretation |
|---:|---:|---|
| 50 | 49 | Actual activity closely followed the plan |
| 50 | 30 | Actual cement demand was below the scheduled amount |
| 50 | 0 | Planned activity did not translate into consumption |
| 50 | 65 | Actual demand exceeded the planned amount |

Exploratory analysis will investigate how strongly planned pours explain actual cement consumption.

## Separating Demand Forecasting from Inventory Management

The project contains two closely related but distinct analytical problems.

### Problem 1 — Demand Forecasting

Estimate future:

`consumed_tonnes`

Potential information includes:

- historical consumption,
- planned pours,
- site,
- cement type,
- weather,
- time and calendar variables.

### Problem 2 — Inventory Management

Use forecasted demand together with:

- opening inventory,
- expected deliveries,
- silo capacity, and
- operational requirements

to estimate future stock levels and determine whether additional cement should be ordered.

The intended analytical flow is therefore:

Historical Data  
↓  
Demand Forecast  
↓  
Inventory Projection  
↓  
Stockout / Overstock Assessment  
↓  
Reorder Decision  
↓  
Management Dashboard

This distinction prevents the forecasting model and inventory decision rules from being unnecessarily mixed into a single model.

## Important Modeling Consideration: Target Leakage

Some variables require special attention before they are used in forecasting.

The inventory relationship is:

Closing Inventory = Opening Inventory + Deliveries - Consumption

Rearranging this relationship gives:

Consumption = Opening Inventory + Deliveries - Closing Inventory

Therefore, using the same day's `closing_inventory_tonnes` as an explanatory variable when predicting the same day's `consumed_tonnes` would reveal information about the target.

A model could appear extremely accurate without genuinely forecasting future demand.

This is known as **target leakage**.

For this reason, variables will later be assessed according to whether the information would genuinely be available at the moment a future forecast is generated.

## Questions to Carry Forward Into Exploratory Data Analysis

The purpose of EDA will not simply be to generate charts.

The analysis should answer operational questions that are relevant to forecasting and inventory management.

### Demand

- How much cement does each site consume?
- Which cement types have the highest demand?
- How different are demand levels across sites?
- How volatile is demand?
- Are there seasonal or recurring temporal patterns?

### Planning

- How closely does actual consumption follow planned pours?
- Which sites have the largest differences between planned and actual demand?

### Weather

- Is cement consumption associated with rainfall?
- Is consumption associated with temperature?
- Does weather help explain deviations from scheduled pours?

### Inventory

- How frequently do sites reach very low or zero inventory?
- How efficiently is silo capacity being utilized?
- Which sites appear to hold excessive inventory?
- Are inventory balance relationships internally consistent?

### Forecasting

- How predictable is historical demand?
- Do different sites require different forecasting approaches?
- Do cement types exhibit different forecasting patterns?
- Is an 8-week forecast sufficiently accurate for operational decision-making?

## Information Availability at Forecast Time

A forecasting model should only use information that would realistically be available when a forecast is generated.

This distinction is important.

### Potentially Known in Advance

- forecast date,
- site,
- cement type,
- planned construction schedule,
- silo capacity.

### Historical Information

- previous cement consumption,
- previous inventories,
- previous deliveries,
- historical weather.

### Information That May Not Be Known Exactly in Advance

- actual future consumption,
- actual future closing inventory,
- actual future rainfall,
- actual future deliveries unless scheduled.

This principle will guide feature selection and prevent data leakage during model development.

## Conceptual Project Architecture

The wider system can be understood as four connected components.

### 1. Demand Forecasting

Historical operational information
+
planned construction activity
+
weather information
+
site and cement characteristics

↓

**Predicted Cement Demand**

---

### 2. Inventory Projection

Predicted Demand
+
Current Inventory
+
Expected Deliveries
+
Silo Capacity

↓

**Projected Future Inventory**

---

### 3. Decision Engine

Projected Inventory

↓

- Stockout risk
- Overstock risk
- Reorder requirement
- Silo utilization

---

### 4. Management Dashboard

Decision outputs

↓

Interactive site-level visibility for operations and procurement managers.

The demand forecast is therefore not the final business output. It is an input into the inventory and procurement decision process.

# Data Understanding Summary

This notebook established the foundation of the MIG Cement Demand Forecasting project.

The SQLite database contains three related tables:

- `Operations`
- `Sites`
- `CementTypes`

The central Operations dataset contains 32,880 records and 11 variables describing daily cement activity across construction sites.

The operational dataset combines four major forms of information:

1. **Demand planning** — planned pour quantities.
2. **Actual demand** — cement consumption.
3. **Inventory and supply** — opening inventory, deliveries and closing inventory.
4. **External and operational conditions** — weather and silo constraints.

The principal forecasting target has been identified as:

**`consumed_tonnes`**

The objective is to estimate future cement consumption and subsequently use those forecasts to support inventory projection and reorder decisions.

An important distinction has also been established between demand forecasting and inventory optimization.

The next stage of the project is therefore:

## Data Cleaning and Validation

This will examine:

- duplicates,
- missing dates,
- invalid numerical values,
- inventory balance integrity,
- silo-capacity violations,
- unusual zero values,
- outliers, and
- consistency across database tables.

Only after the dataset has been validated will full exploratory data analysis and forecasting feature engineering begin.

In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("../data/MIG_Cement_Records.db")

In [3]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,Sites
1,CementTypes
2,Operations


In [4]:
df = pd.read_sql_query(
    "SELECT * FROM Operations",
    conn
)

df.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


In [5]:
52.56 + 45.83 -63.85

34.54

In [6]:
52.56 + 45.83 -34.54

63.85

In [7]:
52.56 + 45.83 - 34.54

63.85

In [8]:
df['cement_type'].value_counts()

cement_type
CEM_II     11097
CEM_I      10913
CEM_III    10870
Name: count, dtype: int64

In [9]:
df.to_csv("../data/dataset.csv", index=False)

In [10]:
site_df = pd.read_sql_query(
    "SELECT * FROM Sites",
    conn
)

In [11]:
site_df.to_csv("../data/site_data.csv", index=False)

In [19]:
site_df

,site_id,region,silo_capacity,behavior
0,SITE_001,North,448,aggressive
1,SITE_002,South,288,conservative
2,SITE_003,East,314,aggressive
3,SITE_004,South,472,conservative
4,SITE_005,South,230,aggressive
5,SITE_006,East,443,chaotic
6,SITE_007,East,485,aggressive
7,SITE_008,West,260,aggressive
8,SITE_009,East,352,conservative
9,SITE_010,West,158,aggressive


In [23]:
#Merge Site table (region and behavior) with Operations table to get the region and behavior for each operation
merged_df = pd.merge(df, site_df[[ 'site_id', 'region', 'behavior']], on='site_id', how='left')


In [24]:
merged_df.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,North,aggressive
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,North,aggressive


In [26]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      32880 non-null  datetime64[us]
 1   site_id                   32880 non-null  str           
 2   cement_type               32880 non-null  str           
 3   planned_pour_tonnes       32880 non-null  float64       
 4   consumed_tonnes           32880 non-null  float64       
 5   opening_inventory_tonnes  32880 non-null  float64       
 6   deliveries_tonnes         32880 non-null  float64       
 7   closing_inventory_tonnes  32880 non-null  float64       
 8   rain_mm                   32880 non-null  float64       
 9   avg_temp_c                32880 non-null  float64       
 10  silo_capacity             32880 non-null  int64         
 11  region                    32880 non-null  str           
 12  behavior                  328

In [27]:
merged_df.to_csv("../data/complete_dataset.csv", index=False)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      32880 non-null  str    
 1   site_id                   32880 non-null  str    
 2   cement_type               32880 non-null  str    
 3   planned_pour_tonnes       32880 non-null  float64
 4   consumed_tonnes           32880 non-null  float64
 5   opening_inventory_tonnes  32880 non-null  float64
 6   deliveries_tonnes         32880 non-null  float64
 7   closing_inventory_tonnes  32880 non-null  float64
 8   rain_mm                   32880 non-null  float64
 9   avg_temp_c                32880 non-null  float64
 10  silo_capacity             32880 non-null  int64  
dtypes: float64(7), int64(1), str(3)
memory usage: 2.8 MB


In [14]:
pd.read_sql_query(
    "SELECT * FROM CementTypes",
    conn
)

,cement_type
0,CEM_I
1,CEM_II
2,CEM_III


In [15]:
site_df=pd.read_sql_query(
    "SELECT * FROM Sites",
    conn
)
site_df.head()

,site_id,region,silo_capacity,behavior
0,SITE_001,North,448,aggressive
1,SITE_002,South,288,conservative
2,SITE_003,East,314,aggressive
3,SITE_004,South,472,conservative
4,SITE_005,South,230,aggressive


#### The major questions will are trying to answer, these questions will drive the whole analysis:

- How much cement does each site consume?

- How does consumption vary by cement type?

- Does cement demand have trends or seasonality?

- How closely does actual consumption follow planned pours?

- Does rain significantly affect consumption?

- Does temperature matter?

- Which sites experience the most volatile demand?

- How frequently do sites approach zero inventory?

- How efficiently are silos being utilized?

- Can historical demand predict future demand?

- Are demand patterns different enough across sites that we need separate models?

- How far ahead can we forecast reliably?

In [16]:
df.groupby("cement_type")["consumed_tonnes"].agg(
    observations="count",
    total_consumption="sum",
    average_daily_consumption="mean",
    median_daily_consumption="median"
)

,observations,total_consumption,average_daily_consumption,median_daily_consumption
cement_type,,,,
CEM_I,10913,259954.71,23.820646,19.79
CEM_II,11097,263418.26,23.737790,19.65
CEM_III,10870,256556.24,23.602230,19.74


In [17]:
df.groupby("site_id")["consumed_tonnes"].agg(
    observations="count",
    total_consumption="sum",
    average_consumption="mean"
).sort_values(
    "total_consumption",
    ascending=False
)

,observations,total_consumption,average_consumption
site_id,,,
SITE_025,1096,33604.06,30.660639
SITE_010,1096,33579.76,30.638467
SITE_018,1096,33348.09,30.427089
SITE_001,1096,33056.40,30.160949
SITE_021,1096,33009.68,30.118321
SITE_005,1096,32935.68,30.050803
SITE_022,1096,32934.03,30.049297
SITE_008,1096,32689.50,29.826186
SITE_007,1096,32607.65,29.751505


In [18]:
# 1. Convert date
df["date"] = pd.to_datetime(df["date"])

# 2. Date range
print("Start:", df["date"].min())
print("End:", df["date"].max())

# 3. Number of sites
print("Number of sites:", df["site_id"].nunique())

# 4. Cement type counts
print(df["cement_type"].value_counts())

# 5. Duplicate site-date-cement records
print(
    "Duplicates:",
    df.duplicated(
        subset=["date", "site_id", "cement_type"]
    ).sum()
)

Start: 2022-01-01 00:00:00
End: 2024-12-31 00:00:00
Number of sites: 30
cement_type
CEM_II     11097
CEM_I      10913
CEM_III    10870
Name: count, dtype: int64
Duplicates: 0


In [30]:
invalid_sites = set(df["site_id"]) - set(site_df["site_id"])

invalid_sites

set()